# 1.缓冲窗口记忆示例

知识点讲解：缓冲窗口记忆（trim_messages + RunnableWithMessageHistory）

1. 什么是缓冲窗口记忆（Buffer Window Memory）
   - 只保留「最近 N 轮」或「最近 N 个 token」的对话，更早的对话直接丢弃
   - 目的：让注入 Prompt 的历史长度恒定可控，避免 token 随轮次无限增长
   - 代价：早期信息被彻底丢弃，模型会对很久以前说过的内容「失忆」
   - 与摘要记忆的区别：窗口记忆「丢弃」旧信息，摘要记忆「压缩」旧信息（见第 11 章）

2. LangChain 1.x 的实现拆分（本示例核心范式）
   旧版由一个 Memory 组件同时负责「存储 + 裁剪 + 注入」，职责混杂且行为隐式。
   1.x 把它拆成三个正交的组件，每个只做一件事：
   - 存储：BaseChatMessageHistory（InMemoryChatMessageHistory / Redis / File...）
   - 裁剪：trim_messages(...)，一个纯函数式的 Runnable
   - 注入：MessagesPlaceholder（Prompt 占位）+ RunnableWithMessageHistory（自动读写）
   好处：可以自由替换任意一层，且每层都能独立单元测试

3. trim_messages 的两种调用形态（关键知识点）
   - 立即执行：trim_messages(messages, max_tokens=..., ...) -> list[BaseMessage]
     第一个位置参数传了消息列表，直接返回裁剪后的结果
   - 延迟执行：trim_messages(max_tokens=..., ...) -> RunnableLambda
     不传消息列表，返回一个 Runnable，可以用 | 接进 LCEL 管道
   本示例用的是第二种，得到的 trimmer 是 RunnableLambda 实例

4. trim_messages 的核心参数详解
   - max_tokens: int        裁剪后允许的最大 token 数（必填）
   - token_counter          token 计数方式，可传：
                              BaseLanguageModel（用模型自带的分词器，最准确）
                              Callable（如 len，按消息条数或字符数近似）
                              "approximate"（内置的快速近似算法）
   - strategy: "first"|"last"
                              "last"  保留最近的消息（缓冲窗口语义，默认）
                              "first" 保留最早的消息（很少用）
   - include_system: bool   是否把 SystemMessage 计入裁剪范围。
                              False 表示 system 消息不参与裁剪，永远保留，
                              这是正确做法（角色设定不能被裁掉）
   - allow_partial: bool    是否允许截断单条消息的内容。
                              False 表示消息要么完整保留、要么整条丢弃
   - start_on: str          裁剪后的第一条消息必须是什么角色。
                              "human" 保证对话以人类消息开头，
                              避免出现以 AIMessage 开头的非法消息序列

5. 为什么 start_on="human" 很重要
   - 大多数模型要求对话序列符合 human/ai 交替出现的格式
   - 若裁剪后恰好以 AIMessage 开头，部分模型（如 Anthropic）会直接报错
   - start_on="human" 会继续往后丢弃，直到第一条是 HumanMessage 为止

6. RunnableWithMessageHistory 的工作机制
   - 把一个普通链包装成「自动带历史」的链，替代旧版的手动 save_context
   - 执行前：按 session_id 调用 get_session_history 取出历史，
             注入到 history_messages_key 指定的键
   - 执行后：自动把本轮的用户输入与 AI 输出 append 回该 session 的历史
   - 三个 key 参数的作用：
       input_messages_key   链输入 dict 中哪个键是「本轮用户输入」
       history_messages_key 链输入 dict 中哪个键用于接收「历史消息列表」
       output_messages_key  链输出 dict 中哪个键是「AI 回复」（输出为 str 时可省略）

7. session_id 与 config 的作用
   - config={"configurable": {"session_id": "xxx"}} 是 LangChain 的运行时配置通道
   - RunnableWithMessageHistory 从中读取 session_id，传给 get_session_history
   - 多用户场景下靠 session_id 隔离记忆，防止串号
   - 忘记传 session_id 会直接抛异常，这是框架的强制约束

8. 本示例的数据流转全过程
   用户输入 "你好"
     ↓ RunnableWithMessageHistory（读历史）
   {"query": "你好", "history": [历史消息列表]}
     ↓ RunnablePassthrough.assign(history=itemgetter("history") | trimmer)
   {"query": "你好", "history": [裁剪后的消息列表]}   ← 注意 history 被覆盖了
     ↓ prompt（MessagesPlaceholder 展开 history）
   [SystemMessage, ...裁剪后的历史..., HumanMessage("你好")]
     ↓ llm → AIMessageChunk 流 → StrOutputParser → str 流
     ↓ RunnableWithMessageHistory（写历史）
   历史中追加 HumanMessage("你好") 与 AIMessage(完整回复)

9. 注意事项
   - assign 中 history 的键名与输入键名相同，会「覆盖」原值，这正是裁剪生效的方式
   - 写回历史的是「未裁剪的完整消息」，裁剪只影响本次发给模型的内容，
     因此 store 中的历史会持续增长，裁剪只是发送时的过滤
   - token_counter=llm 会调用模型的分词器，对部分模型可能产生额外开销
   - InMemoryChatMessageHistory 进程退出即丢失，生产环境请换 Redis 实现


In [2]:
from operator import itemgetter
import dotenv
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.messages import trim_messages
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是DeepSeek开发的聊天机器人，请根据对应的上下文回复用户问题"),
    MessagesPlaceholder("history"),
    ("human", "{query}"),
])
llm = ChatOpenAI(model="deepseek-v4-pro")
trimmer = trim_messages(
    max_tokens=200,
    strategy="last",
    token_counter=llm,
    include_system=False,
    allow_partial=False,
    start_on="human",
)
chain = RunnablePassthrough.assign(
    history=itemgetter("history") | trimmer
) | prompt | llm | StrOutputParser()
store: dict[str, BaseChatMessageHistory] = {}

In [3]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """
    get_session_history(session_id: str) -> BaseChatMessageHistory
      作用：会话历史工厂函数，根据 session_id 获取（或懒创建）对应的历史存储
      参数：
        session_id: str - 会话唯一标识，由 config["configurable"]["session_id"] 传入
      返回：
        BaseChatMessageHistory - 该会话的消息历史存储实例
      调用时机：
        由 RunnableWithMessageHistory 在每次 invoke/stream 时自动调用，
        业务代码无需手动调用（本示例末尾的调用仅用于打印观察）
      设计要点：
        1. 懒加载：首次访问某个 session_id 时才创建实例，避免预分配浪费
        2. 幂等：同一个 session_id 必须始终返回「同一个实例」，
                 否则历史无法累积（每次都是新的空历史）
        3. 可替换：只需改这里的构造函数即可切换存储后端，例如
                   return RedisChatMessageHistory(session_id, url="redis://...")
    """
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [5]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="query",
    history_messages_key="history",
)
config = {"configurable": {"session_id": "muxiaoke"}}
while True:
    query = input("Human: ")
    if query == "q":
        exit(0)
    response = chain_with_history.stream({"query": query}, config=config)
    print("AI: ", flush=True, end="")
    for chunk in response:
        print(chunk, flush=True, end="")
    print("")
    print("history: ", get_session_history("muxiaoke").messages)

KeyboardInterrupt: Interrupted by user